In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount("/content/drive")

GITHUB_REPO = "https://github.com/imnima2005/scrubref-unlearning.git"
REPO_DIR_NAME = "scrubref-unlearning"

DRIVE_ROOT = Path("/content/drive/MyDrive")
REPO_DIR = DRIVE_ROOT / REPO_DIR_NAME

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", GITHUB_REPO, str(REPO_DIR)],
        check=True
    )
else:
    print(f"already exist: {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Project root:", Path.cwd())

Mounted at /content/drive
already exist: /content/drive/MyDrive/scrubref-unlearning
Project root: /content/drive/MyDrive/scrubref-unlearning


In [ ]:
# Colab environment check — DO NOT install or downgrade core packages in this notebook.
import importlib.metadata as metadata
import platform
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for package in ("numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "tensorflow", "joblib"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("Environment check complete. Continue to the import cell.")


Python: 3.13.16
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
numpy: 2.4.6
pandas: 2.2.3
scipy: 1.16.3
scikit-learn: 1.6.1
matplotlib: 3.10.0
tensorflow: 2.21.0
joblib: 1.6.0
Environment check complete. Continue to the import cell.


In [ ]:
from __future__ import annotations

import ctypes
import gc
import json
import math
import os
import random
import shutil
import tarfile
import time
import urllib.request
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple, Union

try:
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
except ValueError as exc:
    if "numpy.dtype size changed" in str(exc):
        raise RuntimeError(
            "The Colab VM contains an incompatible NumPy/Pandas binary stack. "
            "Use Runtime > Disconnect and delete runtime, reconnect, and run this notebook "
            "without executing any pip-install cell."
        ) from exc
    raise
from sklearn import set_config as sklearn_set_config
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

sklearn_set_config(working_memory=256)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import shutil
from pathlib import Path
import os

# Configure persistent Keras datasets directory directly in your project's data folder
DRIVE_DATASETS = REPO_DIR / "data"
LOCAL_DATASETS = Path.home() / ".keras" / "datasets"

DRIVE_DATASETS.mkdir(parents=True, exist_ok=True)
LOCAL_DATASETS.parent.mkdir(parents=True, exist_ok=True)

# Replace only the ephemeral local cache directory with a Drive-backed symlink
if LOCAL_DATASETS.is_symlink():
    if LOCAL_DATASETS.resolve() != DRIVE_DATASETS.resolve():
        LOCAL_DATASETS.unlink()
elif LOCAL_DATASETS.exists():
    shutil.rmtree(LOCAL_DATASETS)

if not LOCAL_DATASETS.exists():
    os.symlink(str(DRIVE_DATASETS), str(LOCAL_DATASETS), target_is_directory=True)

os.environ["KERAS_HOME"] = str(REPO_DIR)

print("Persistent Keras cache:", DRIVE_DATASETS)
print("Local cache link:", LOCAL_DATASETS, "->", LOCAL_DATASETS.resolve())

Persistent Keras cache: /content/drive/MyDrive/scrubref-unlearning/data
Local cache link: /root/.keras/datasets -> /content/drive/MyDrive/scrubref-unlearning/data


In [ ]:
# Prepare CIFAR-100 from a fast mirror and keep it permanently in Google Drive.
from pathlib import Path
from src.utils import file_md5
import hashlib
import shutil
import subprocess
import tarfile

# Use the same DRIVE_DATASETS path (REPO_DIR / "data") configured in the symlink cell
main_dataset_dir = REPO_DIR / "data"
dataset_dir = main_dataset_dir / "cifar100"
dataset_dir.mkdir(parents=True, exist_ok=True)

local_archive = Path("/content/cifar-100-python.tar.gz")
drive_archive = dataset_dir / "cifar-100-python.tar.gz"
extracted_dir = dataset_dir / "cifar-100-python"
expected_md5 = "eb9058c3a382ffc7106e4002c42a8d85"

mirror_urls = [
    "https://huggingface.co/datasets/nakroy/cifar100-python/resolve/main/cifar-100-python.tar.gz",
    "https://data.brainchip.com/dataset-mirror/cifar100/cifar-100-python.tar.gz",
]

required_extracted_files = [
    extracted_dir / "train",
    extracted_dir / "test",
    extracted_dir / "meta",
]

if all(path.exists() for path in required_extracted_files):
    print("CIFAR-100 already prepared:", extracted_dir)
else:
    if drive_archive.exists() and file_md5(drive_archive) != expected_md5:
        print("Removing invalid Drive archive:", drive_archive)
        drive_archive.unlink()

    if not drive_archive.exists():
        local_archive.unlink(missing_ok=True)
        downloaded = False
        for url in mirror_urls:
            print("Trying:", url)
            result = subprocess.run(
                [
                    "wget",
                    "-c",
                    "--progress=bar:force:noscroll",
                    "--timeout=30",
                    "--tries=3",
                    "-O",
                    str(local_archive),
                    url,
                ],
                check=False,
            )
            if result.returncode != 0 or not local_archive.exists():
                continue
            actual_md5 = file_md5(local_archive)
            print("MD5:", actual_md5)
            if actual_md5 == expected_md5:
                shutil.copy2(local_archive, drive_archive)
                downloaded = True
                break
            local_archive.unlink(missing_ok=True)

        if not downloaded:
            raise RuntimeError("All CIFAR-100 mirrors failed or returned an invalid archive.")

    print("Extracting CIFAR-100 into Google Drive...")
    with tarfile.open(drive_archive, "r:gz") as archive:
        try:
            archive.extractall(dataset_dir, filter="data")
        except TypeError:
            archive.extractall(dataset_dir)

    missing = [str(path) for path in required_extracted_files if not path.exists()]
    if missing:
        raise RuntimeError("CIFAR-100 extraction is incomplete:\n" + "\n".join(missing))
    print("Dataset prepared:", extracted_dir)

local_archive.unlink(missing_ok=True)

CIFAR-100 already prepared: /content/drive/MyDrive/scrubref-unlearning/data/cifar100/cifar-100-python


In [ ]:
from src.config import ExperimentConfig

CFG = ExperimentConfig.from_yaml(REPO_DIR / "configs" / "imagewoof.yaml")

CFG.run_mode = "debug"
CFG.apply_mode()

In [ ]:
from src.utils import (
    array_memory_mb,
    release_memory,
    reset_tensorflow,
    set_global_determinism,
)

set_global_determinism(CFG.seeds[0])

In [ ]:
ArrayLike = Union[np.ndarray, Sequence[str]]


@dataclass
class DataBundle:
    name: str
    train_items: np.ndarray
    train_labels: np.ndarray
    test_items: np.ndarray
    test_labels: np.ndarray
    class_names: List[str]
    image_size: Tuple[int, int]
    channels: int = 3
    path_mode: bool = False

    @property
    def num_classes(self) -> int:
        return len(self.class_names)

    @property
    def input_shape(self) -> Tuple[int, int, int]:
        return self.image_size + (self.channels,)


@dataclass
class UnlearningSplit:
    retain_train_items: np.ndarray
    retain_train_labels: np.ndarray
    retain_val_items: np.ndarray
    retain_val_labels: np.ndarray
    forget_train_items: np.ndarray
    forget_train_labels: np.ndarray
    retain_test_items: np.ndarray
    retain_test_labels: np.ndarray
    forget_test_items: np.ndarray
    forget_test_labels: np.ndarray


def ensure_imagewoof(root: str) -> Path:
    root_path = Path(root)
    dataset_dir = root_path / "imagewoof2-320"
    if dataset_dir.exists():
        return dataset_dir
    if not CFG.download_imagewoof:
        raise FileNotFoundError(f"ImageWoof not found at {dataset_dir}")
    root_path.mkdir(parents=True, exist_ok=True)
    archive = root_path / "imagewoof2-320.tgz"
    url = "https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2-320.tgz"
    if not archive.exists():
        urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(root_path)
    return dataset_dir


def list_directory_split(split_dir: Path, class_names: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    paths: List[str] = []
    labels: List[int] = []
    valid_suffixes = {".jpg", ".jpeg", ".png", ".webp"}
    for class_id, class_name in enumerate(class_names):
        class_dir = split_dir / class_name
        files = sorted(p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in valid_suffixes)
        paths.extend(str(p) for p in files)
        labels.extend([class_id] * len(files))
    return np.asarray(paths, dtype=str), np.asarray(labels, dtype=np.int64)


def load_directory_bundle(name: str, root: str, image_size: Tuple[int, int]) -> DataBundle:
    root_path = Path(root)
    train_dir = root_path / "train"
    val_dir = root_path / "val"
    if not train_dir.exists() or not val_dir.exists():
        raise FileNotFoundError(f"Expected {train_dir} and {val_dir}")
    class_names = sorted(p.name for p in train_dir.iterdir() if p.is_dir())
    train_items, train_labels = list_directory_split(train_dir, class_names)
    test_items, test_labels = list_directory_split(val_dir, class_names)
    if len(train_items) == 0 or len(test_items) == 0:
        raise RuntimeError(f"No images were found under {root_path}")
    return DataBundle(
        name=name,
        train_items=train_items,
        train_labels=train_labels,
        test_items=test_items,
        test_labels=test_labels,
        class_names=class_names,
        image_size=image_size,
        path_mode=True,
    )


def load_dataset(cfg: ExperimentConfig) -> DataBundle:
    if cfg.dataset == "cifar10":
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
        return DataBundle(
            name="cifar10",
            train_items=x_train,
            train_labels=y_train.reshape(-1).astype(np.int64),
            test_items=x_test,
            test_labels=y_test.reshape(-1).astype(np.int64),
            class_names=["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"],
            image_size=(32, 32),
            path_mode=False,
        )
    if cfg.dataset == "cifar100":
        import pickle

        dataset_dir = Path(
            "./data/cifar100/cifar-100-python" if IN_COLAB else REPO_DIR / "data/cifar100/cifar-100-python"
        )

        train_file = dataset_dir / "train"
        test_file = dataset_dir / "test"
        meta_file = dataset_dir / "meta"

        required_files = [train_file, test_file, meta_file]
        missing_files = [str(path) for path in required_files if not path.exists()]

        if missing_files:
            raise FileNotFoundError(
                "Missing CIFAR-100 files:\n" + "\n".join(missing_files)
            )

        def load_cifar100_pickle(path: Path):
            with path.open("rb") as file:
                payload = pickle.load(file, encoding="bytes")

            data = payload.get(b"data", payload.get("data"))
            labels = payload.get(
                b"fine_labels",
                payload.get("fine_labels"),
            )

            if data is None or labels is None:
                raise ValueError(f"Invalid CIFAR-100 file: {path}")

            images = np.asarray(data, dtype=np.uint8)
            images = images.reshape(-1, 3, 32, 32)
            images = images.transpose(0, 2, 3, 1)

            labels = np.asarray(labels, dtype=np.int64)

            return images, labels

        x_train, y_train = load_cifar100_pickle(train_file)
        x_test, y_test = load_cifar100_pickle(test_file)

        with meta_file.open("rb") as file:
            meta = pickle.load(file, encoding="bytes")

        raw_class_names = meta.get(
            b"fine_label_names",
            meta.get("fine_label_names"),
        )

        class_names = [
            name.decode("utf-8") if isinstance(name, bytes) else str(name)
            for name in raw_class_names
        ]

        print("Loaded CIFAR-100 directly from Google Drive")
        print("Train:", x_train.shape, y_train.shape)
        print("Test:", x_test.shape, y_test.shape)

        return DataBundle(
            name="cifar100",
            train_items=x_train,
            train_labels=y_train,
            test_items=x_test,
            test_labels=y_test,
            class_names=class_names,
            image_size=(32, 32),
            path_mode=False,
        )
    if cfg.dataset == "imagewoof":
        dataset_dir = ensure_imagewoof(cfg.data_root)
        bundle = load_directory_bundle("imagewoof", str(dataset_dir), (224, 224))
        configured = cfg.forget_classes_by_dataset["imagewoof"]
        if configured == (0,) and "n02086240" in bundle.class_names:
            cfg.forget_classes_by_dataset["imagewoof"] = (bundle.class_names.index("n02086240"),)
        return bundle
    if cfg.dataset == "imagenet100":
        return load_directory_bundle("imagenet100", cfg.imagenet100_root, (224, 224))
    raise ValueError(f"Unsupported dataset: {cfg.dataset}")


def make_unlearning_split(
    bundle: DataBundle,
    forget_classes: Sequence[int],
    validation_fraction: float,
    seed: int,
) -> UnlearningSplit:
    forget_classes_array = np.asarray(forget_classes, dtype=np.int64)
    train_forget_mask = np.isin(bundle.train_labels, forget_classes_array)
    test_forget_mask = np.isin(bundle.test_labels, forget_classes_array)

    retain_train_pool_indices = np.flatnonzero(~train_forget_mask)
    forget_train_indices = np.flatnonzero(train_forget_mask)
    retain_test_indices = np.flatnonzero(~test_forget_mask)
    forget_test_indices = np.flatnonzero(test_forget_mask)

    retain_train_indices, retain_val_indices = train_test_split(
        retain_train_pool_indices,
        test_size=validation_fraction,
        random_state=seed,
        stratify=bundle.train_labels[retain_train_pool_indices],
    )

    split = UnlearningSplit(
        retain_train_items=bundle.train_items[retain_train_indices],
        retain_train_labels=bundle.train_labels[retain_train_indices],
        retain_val_items=bundle.train_items[retain_val_indices],
        retain_val_labels=bundle.train_labels[retain_val_indices],
        forget_train_items=bundle.train_items[forget_train_indices],
        forget_train_labels=bundle.train_labels[forget_train_indices],
        retain_test_items=bundle.test_items[retain_test_indices],
        retain_test_labels=bundle.test_labels[retain_test_indices],
        forget_test_items=bundle.test_items[forget_test_indices],
        forget_test_labels=bundle.test_labels[forget_test_indices],
    )

    print(
        "RAM-safe split prepared:",
        f"retain_train={len(split.retain_train_labels)},",
        f"retain_val={len(split.retain_val_labels)},",
        f"forget_train={len(split.forget_train_labels)},",
        f"retain_test={len(split.retain_test_labels)},",
        f"forget_test={len(split.forget_test_labels)}",
    )
    return split


def decode_item(item: tf.Tensor, image_size: Tuple[int, int], path_mode: bool) -> tf.Tensor:
    if path_mode:
        raw = tf.io.read_file(item)
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image = tf.image.resize(image, image_size, antialias=True)
    else:
        image = tf.cast(item, tf.float32)
        if tuple(image.shape[:2]) != image_size:
            image = tf.image.resize(image, image_size, antialias=True)
    image.set_shape(image_size + (3,))
    return tf.cast(image, tf.float32)


def augment_image(image: tf.Tensor, image_size: Tuple[int, int]) -> tf.Tensor:
    image = tf.image.random_flip_left_right(image)
    if image_size[0] <= 64:
        pad = 4
        image = tf.image.resize_with_crop_or_pad(image, image_size[0] + 2 * pad, image_size[1] + 2 * pad)
        image = tf.image.random_crop(image, image_size + (3,))
    else:
        enlarged = (int(image_size[0] * 1.08), int(image_size[1] * 1.08))
        image = tf.image.resize(image, enlarged, antialias=True)
        image = tf.image.random_crop(image, image_size + (3,))
    image = tf.image.random_brightness(image, max_delta=8.0)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    return tf.clip_by_value(image, 0.0, 255.0)


def make_dataset(
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    batch_size: int,
    shuffle: bool,
    augment: bool,
    repeat: bool = False,
    seed: int = 42,
) -> tf.data.Dataset:
    if len(items) != len(labels):
        raise ValueError(f"items/labels length mismatch: {len(items)} != {len(labels)}")

    labels_array = np.asarray(labels, dtype=np.int64)
    ds = tf.data.Dataset.from_tensor_slices((items, labels_array))
    if shuffle:
        ds = ds.shuffle(
            min(len(items), CFG.shuffle_buffer),
            seed=seed,
            reshuffle_each_iteration=True,
        )
    if repeat:
        ds = ds.repeat()

    def mapper(item: tf.Tensor, label: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
        image = decode_item(item, bundle.image_size, bundle.path_mode)
        if augment:
            image = augment_image(image, bundle.image_size)
        return image, tf.cast(label, tf.int64)

    options = tf.data.Options()
    options.experimental_deterministic = not shuffle
    try:
        options.autotune.enabled = False
        options.threading.private_threadpool_size = CFG.data_private_threadpool_size
        options.threading.max_intra_op_parallelism = 1
    except Exception:
        pass

    ds = ds.with_options(options)
    ds = ds.map(mapper, num_parallel_calls=CFG.data_parallel_calls)
    ds = ds.batch(batch_size, drop_remainder=False)
    return ds.prefetch(CFG.data_prefetch_batches)


def materialize_items(
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    limit: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    if len(items) > limit:
        indices = rng.choice(len(items), size=limit, replace=False)
        items = items[indices]
        labels = labels[indices]
    ds = make_dataset(items, labels, bundle, CFG.eval_batch_size, False, False, False, seed)
    images_out: List[np.ndarray] = []
    labels_out: List[np.ndarray] = []
    for xb, yb in ds:
        images_out.append(xb.numpy())
        labels_out.append(yb.numpy())
    return np.concatenate(images_out), np.concatenate(labels_out)

## Model construction and output handling

Loss functions receive the correct `from_logits` mode. The notebook infers the mode from the final activation and validates it on a batch unless a manual override is supplied.

In [ ]:
def residual_block(x: tf.Tensor, filters: int, stride: int, name: str) -> tf.Tensor:
    shortcut = x
    x = tf.keras.layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False, name=f"{name}_conv1")(x)
    x = tf.keras.layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = tf.keras.layers.ReLU(name=f"{name}_relu1")(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(x)
    x = tf.keras.layers.BatchNormalization(name=f"{name}_bn2")(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, strides=stride, use_bias=False, name=f"{name}_proj_conv")(shortcut)
        shortcut = tf.keras.layers.BatchNormalization(name=f"{name}_proj_bn")(shortcut)
    x = tf.keras.layers.Add(name=f"{name}_add")([x, shortcut])
    return tf.keras.layers.ReLU(name=f"{name}_out")(x)


def build_resnet18(input_shape: Tuple[int, int, int], num_classes: int) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=input_shape, name="image")
    x = tf.keras.layers.Rescaling(1.0 / 255.0, name="rescale")(inputs)
    if input_shape[0] <= 64:
        x = tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False, name="stem_conv")(x)
    else:
        x = tf.keras.layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False, name="stem_conv")(x)
    x = tf.keras.layers.BatchNormalization(name="stem_bn")(x)
    x = tf.keras.layers.ReLU(name="stem_relu")(x)
    if input_shape[0] > 64:
        x = tf.keras.layers.MaxPool2D(3, strides=2, padding="same", name="stem_pool")(x)
    filters_per_stage = [64, 128, 256, 512]
    for stage, filters in enumerate(filters_per_stage):
        x = residual_block(x, filters, 1 if stage == 0 else 2, f"stage{stage + 1}_block1")
        x = residual_block(x, filters, 1, f"stage{stage + 1}_block2")
    x = tf.keras.layers.GlobalAveragePooling2D(name="penultimate_features")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation=None, dtype="float32", name="classifier")(x)
    return tf.keras.Model(inputs, outputs, name="resnet18_scrubref")


def custom_objects() -> Dict[str, object]:
    return {
        "preprocess_input": tf.keras.applications.efficientnet.preprocess_input,
    }


def clone_architecture(model: tf.keras.Model) -> tf.keras.Model:
    tf.keras.utils.get_custom_objects().update(custom_objects())
    try:
        return tf.keras.models.clone_model(model)
    except Exception as first_error:
        try:
            return tf.keras.models.model_from_json(model.to_json(), custom_objects=custom_objects())
        except Exception as second_error:
            raise RuntimeError(
                "The checkpoint architecture could not be cloned. Save a serialization-safe .keras model "
                "or remove unsupported Lambda layers before running the revision pipeline."
            ) from second_error


def clone_with_weights(model: tf.keras.Model) -> tf.keras.Model:
    cloned = clone_architecture(model)
    cloned.set_weights(model.get_weights())
    return cloned


def fresh_clone(model: tf.keras.Model, initial_weights_path: str = "") -> tf.keras.Model:
    cloned = clone_architecture(model)
    if initial_weights_path and Path(initial_weights_path).exists():
        cloned.load_weights(initial_weights_path)
    return cloned


def output_to_logits(output: tf.Tensor, output_is_logits: bool) -> tf.Tensor:
    output = tf.cast(output, tf.float32)
    if output_is_logits:
        return output
    probabilities = tf.clip_by_value(output, 1e-7, 1.0)
    return tf.math.log(probabilities)


def output_to_probs(output: tf.Tensor, output_is_logits: bool, temperature: float = 1.0) -> tf.Tensor:
    logits = output_to_logits(output, output_is_logits) / temperature
    return tf.nn.softmax(logits, axis=-1)


def infer_output_is_logits(model: tf.keras.Model, sample_batch: tf.Tensor, override: Optional[bool]) -> bool:
    if override is not None:
        return bool(override)
    last_layer = model.layers[-1]
    activation = getattr(last_layer, "activation", None)
    activation_name = getattr(activation, "__name__", "")
    if activation_name == "softmax":
        return False
    if activation_name == "linear":
        return True
    output = tf.cast(model(sample_batch[:8], training=False), tf.float32)
    nonnegative = bool(tf.reduce_all(output >= 0.0).numpy())
    sums_close = bool(tf.reduce_all(tf.abs(tf.reduce_sum(output, axis=-1) - 1.0) < 1e-3).numpy())
    return not (nonnegative and sums_close)


def leaf_layers(model: tf.keras.Model) -> List[tf.keras.layers.Layer]:
    result: List[tf.keras.layers.Layer] = []
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            result.extend(leaf_layers(layer))
        else:
            result.append(layer)
    return result


def model_containers(model: tf.keras.Model) -> List[tf.keras.Model]:
    containers = [model]
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            containers.extend(model_containers(layer))
    return containers


def classifier_layer(model: tf.keras.Model, num_classes: int) -> tf.keras.layers.Dense:
    candidates = [layer for layer in leaf_layers(model) if isinstance(layer, tf.keras.layers.Dense) and layer.units == num_classes]
    if not candidates:
        raise ValueError(f"No Dense classifier with {num_classes} units was found.")
    return candidates[-1]


def build_feature_model(model: tf.keras.Model, num_classes: int) -> tf.keras.Model:
    classifier = classifier_layer(model, num_classes)
    return tf.keras.Model(model.inputs, classifier.input, name=f"{model.name}_feature_model")


def set_all_trainable(model: tf.keras.Model, trainable: bool = True, freeze_batch_norm: bool = False) -> None:
    for container in model_containers(model):
        container.trainable = True
    for layer in leaf_layers(model):
        if freeze_batch_norm and isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = trainable


def set_tail_trainable(model: tf.keras.Model, num_classes: int, tail_weight_layers: int) -> List[str]:
    for container in model_containers(model):
        container.trainable = True
    leaves = leaf_layers(model)
    for layer in leaves:
        layer.trainable = False
    classifier = classifier_layer(model, num_classes)
    classifier.trainable = True
    candidates = [
        layer for layer in leaves
        if layer is not classifier
        and layer.weights
        and not isinstance(layer, tf.keras.layers.BatchNormalization)
    ]
    selected = candidates[-tail_weight_layers:] if tail_weight_layers > 0 else []
    for layer in selected:
        layer.trainable = True
    return [layer.name for layer in selected] + [classifier.name]


def compile_supervised(model: tf.keras.Model, output_is_logits: bool, learning_rate: float) -> None:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=output_is_logits),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )


def train_supervised(
    model: tf.keras.Model,
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    output_is_logits: bool,
    epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    validation: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    train_all: bool = True,
    tail_layers: int = 0,
) -> tf.keras.Model:
    if train_all:
        set_all_trainable(model, True, freeze_batch_norm=False)
    else:
        set_tail_trainable(model, bundle.num_classes, tail_layers)
    compile_supervised(model, output_is_logits, learning_rate)
    train_ds = make_dataset(items, labels, bundle, batch_size, True, True, False, seed)
    validation_ds = None
    callbacks: List[tf.keras.callbacks.Callback] = []
    if validation is not None and len(validation[0]) > 0:
        validation_ds = make_dataset(validation[0], validation[1], bundle, batch_size, False, False, False, seed)
        callbacks.append(tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=max(3, epochs // 10),
            restore_best_weights=True,
        ))
    model.fit(train_ds, validation_data=validation_ds, epochs=epochs, verbose=2, callbacks=callbacks)
    return model


def load_or_train_original(
    cfg: ExperimentConfig,
    bundle: DataBundle,
    seed: int,
    run_dir: Path,
) -> Tuple[tf.keras.Model, bool, str]:
    checkpoint = Path(cfg.checkpoint_paths[cfg.dataset])
    if checkpoint.exists():
        model = tf.keras.models.load_model(checkpoint, custom_objects=custom_objects(), compile=False, safe_mode=False)
        source = str(checkpoint)
    else:
        model = build_resnet18(bundle.input_shape, bundle.num_classes)
        sample_ds = make_dataset(bundle.train_items[:16], bundle.train_labels[:16], bundle, 16, False, False, False, seed)
        sample_x, _ = next(iter(sample_ds))
        output_is_logits = infer_output_is_logits(model, sample_x, True)
        model = train_supervised(
            model,
            bundle.train_items,
            bundle.train_labels,
            bundle,
            output_is_logits,
            cfg.original_epochs,
            cfg.batch_size,
            cfg.original_lr,
            seed,
            validation=None,
            train_all=True,
        )
        source = str(f"./checkpoints/{cfd.dataset}_trained_original.keras")
        model.save(source)
    sample_ds = make_dataset(bundle.train_items[:16], bundle.train_labels[:16], bundle, 16, False, False, False, seed)
    sample_x, _ = next(iter(sample_ds))
    output_is_logits = infer_output_is_logits(model, sample_x, cfg.output_is_logits)
    return model, output_is_logits, source

## Metrics and statistical evaluation

The membership attack is class-conditional: members are forgotten-class training examples and non-members are forgotten-class test examples. This replaces the invalid forgotten-class-versus-other-class AUC used in the submitted notebook.

In [ ]:
def predict_probabilities(
    model: tf.keras.Model,
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    output_is_logits: bool,
    batch_size: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    ds = make_dataset(items, labels, bundle, batch_size, False, False, False, seed)
    probabilities: List[np.ndarray] = []
    labels_out: List[np.ndarray] = []
    for xb, yb in ds:
        output = model(xb, training=False)
        probabilities.append(output_to_probs(output, output_is_logits).numpy())
        labels_out.append(yb.numpy())
    return np.concatenate(probabilities), np.concatenate(labels_out)


def expected_calibration_error(y_true: np.ndarray, probs: np.ndarray, bins: int = 15) -> float:
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correctness = (predictions == y_true).astype(np.float64)
    boundaries = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for lower, upper in zip(boundaries[:-1], boundaries[1:]):
        mask = (confidences > lower) & (confidences <= upper)
        if np.any(mask):
            ece += np.abs(correctness[mask].mean() - confidences[mask].mean()) * mask.mean()
    return float(ece)


def brier_multiclass(y_true: np.ndarray, probs: np.ndarray, num_classes: int) -> float:
    one_hot = np.eye(num_classes, dtype=np.float64)[y_true]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def per_class_accuracy(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int) -> np.ndarray:
    result = np.full(num_classes, np.nan, dtype=np.float64)
    for class_id in range(num_classes):
        mask = y_true == class_id
        if np.any(mask):
            result[class_id] = np.mean(y_pred[mask] == class_id)
    return result


def evaluate_utility(
    model: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    batch_size: int,
    seed: int,
) -> Tuple[Dict[str, float], np.ndarray]:
    probs, y_true = predict_probabilities(
        model, bundle.test_items, bundle.test_labels, bundle, output_is_logits, batch_size, seed
    )
    y_pred = probs.argmax(axis=1)
    forget_mask = np.isin(y_true, np.asarray(forget_classes))
    retain_mask = ~forget_mask
    top5 = np.mean([
        label in np.argpartition(row, -min(5, bundle.num_classes))[-min(5, bundle.num_classes):]
        for row, label in zip(probs, y_true)
    ])
    metrics = {
        "overall_accuracy": float(accuracy_score(y_true, y_pred)),
        "retained_accuracy": float(accuracy_score(y_true[retain_mask], y_pred[retain_mask])),
        "forgotten_accuracy": float(accuracy_score(y_true[forget_mask], y_pred[forget_mask])),
        "top5_accuracy": float(top5),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "nll": float(log_loss(y_true, np.clip(probs, 1e-7, 1.0), labels=np.arange(bundle.num_classes))),
        "brier": brier_multiclass(y_true, probs, bundle.num_classes),
        "ece_15": expected_calibration_error(y_true, probs, bins=15),
        "mean_global_forget_mass": float(np.mean(np.sum(probs[:, forget_classes], axis=1))),
        "mean_forget_test_true_class_confidence": float(
            np.mean(probs[np.where(forget_mask)[0], y_true[forget_mask]])
        ),
    }
    return metrics, per_class_accuracy(y_true, y_pred, bundle.num_classes)


def entropy_scores(probs: np.ndarray) -> np.ndarray:
    probs = np.clip(probs, 1e-7, 1.0 - 1e-7)
    return -np.sum(probs * np.log(probs), axis=1)


def modified_entropy_scores(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
    probs = np.clip(probs, 1e-7, 1.0 - 1e-7)
    rows = np.arange(len(labels))
    true_probs = probs[rows, labels]
    first = -(1.0 - true_probs) * np.log(true_probs)
    mask = np.ones_like(probs, dtype=bool)
    mask[rows, labels] = False
    remaining = -np.sum((probs * np.log(1.0 - probs)) * mask, axis=1)
    return first + remaining


def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    auc = roc_auc_score(y_true, scores)
    return float(max(auc, 1.0 - auc))


def learned_mia(
    member_probs: np.ndarray,
    member_labels: np.ndarray,
    nonmember_probs: np.ndarray,
    nonmember_labels: np.ndarray,
    repeats: int,
    seed: int,
) -> Tuple[float, float]:
    def attack_features(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
        rows = np.arange(len(labels))
        true_conf = probs[rows, labels]
        entropy = entropy_scores(probs)
        mod_entropy = modified_entropy_scores(probs, labels)
        sorted_probs = np.sort(probs, axis=1)[:, -3:]
        return np.column_stack([true_conf, entropy, mod_entropy, sorted_probs])

    x = np.concatenate([
        attack_features(member_probs, member_labels),
        attack_features(nonmember_probs, nonmember_labels),
    ])
    y = np.concatenate([
        np.ones(len(member_probs), dtype=np.int64),
        np.zeros(len(nonmember_probs), dtype=np.int64),
    ])
    aucs: List[float] = []
    aps: List[float] = []
    for repeat in range(repeats):
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.5, random_state=seed + repeat, stratify=y
        )
        attack = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
        attack.fit(x_train, y_train)
        scores = attack.predict_proba(x_test)[:, 1]
        aucs.append(roc_auc_score(y_test, scores))
        aps.append(average_precision_score(y_test, scores))
    return float(np.mean(aucs)), float(np.mean(aps))


def evaluate_membership_inference(
    model: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    batch_size: int,
    repeats: int,
    seed: int,
) -> Dict[str, float]:
    member_probs, member_labels = predict_probabilities(
        model,
        split.forget_train_items,
        split.forget_train_labels,
        bundle,
        output_is_logits,
        batch_size,
        seed,
    )
    nonmember_probs, nonmember_labels = predict_probabilities(
        model,
        split.forget_test_items,
        split.forget_test_labels,
        bundle,
        output_is_logits,
        batch_size,
        seed,
    )
    n = min(len(member_probs), len(nonmember_probs))
    rng = np.random.default_rng(seed)
    member_idx = rng.choice(len(member_probs), n, replace=False)
    nonmember_idx = rng.choice(len(nonmember_probs), n, replace=False)
    member_probs = member_probs[member_idx]
    member_labels = member_labels[member_idx]
    nonmember_probs = nonmember_probs[nonmember_idx]
    nonmember_labels = nonmember_labels[nonmember_idx]

    probs = np.concatenate([member_probs, nonmember_probs])
    labels = np.concatenate([member_labels, nonmember_labels])
    membership = np.concatenate([np.ones(n), np.zeros(n)])
    rows = np.arange(len(labels))
    true_conf = probs[rows, labels]
    negative_loss = np.log(np.clip(true_conf, 1e-7, 1.0))
    negative_entropy = -entropy_scores(probs)
    negative_modified_entropy = -modified_entropy_scores(probs, labels)
    learned_auc, learned_ap = learned_mia(
        member_probs, member_labels, nonmember_probs, nonmember_labels, repeats, seed
    )
    return {
        "mia_confidence_auc": safe_auc(membership, true_conf),
        "mia_loss_auc": safe_auc(membership, negative_loss),
        "mia_entropy_auc": safe_auc(membership, negative_entropy),
        "mia_modified_entropy_auc": safe_auc(membership, negative_modified_entropy),
        "mia_learned_auc": learned_auc,
        "mia_learned_ap": learned_ap,
        "mia_member_count": float(n),
    }


def parameter_change_statistics(original: tf.keras.Model, changed: tf.keras.Model, tolerance: float = 1e-7) -> Dict[str, float]:
    original_weights = original.get_weights()
    changed_weights = changed.get_weights()
    if len(original_weights) != len(changed_weights):
        return {"changed_parameter_fraction": float("nan"), "relative_l2_parameter_change": float("nan")}
    total = 0
    changed_count = 0
    squared_delta = 0.0
    squared_original = 0.0
    for before, after in zip(original_weights, changed_weights):
        delta = after.astype(np.float64) - before.astype(np.float64)
        total += delta.size
        changed_count += int(np.count_nonzero(np.abs(delta) > tolerance))
        squared_delta += float(np.sum(delta ** 2))
        squared_original += float(np.sum(before.astype(np.float64) ** 2))
    return {
        "changed_parameter_fraction": float(changed_count / max(total, 1)),
        "relative_l2_parameter_change": float(math.sqrt(squared_delta) / max(math.sqrt(squared_original), 1e-12)),
    }


def peak_gpu_memory_mb() -> float:
    try:
        info = tf.config.experimental.get_memory_info("GPU:0")
        return float(info["peak"] / (1024 ** 2))
    except Exception:
        return float("nan")

## Representation audits

The audit measures whether forgotten classes remain recoverable from the penultimate representation. CKA is computed in feature space, avoiding the quadratic sample-by-sample Gram matrix.

## Synthetic proxy generation

The original notebook assigned zero class weight to every synthetic sample, so the scrub branch contributed zero loss. Here, proxy samples are optimized through the frozen original model and the notebook asserts that the forget branch produces a non-zero gradient.

In [ ]:
def balanced_proxy_targets(forget_classes: Sequence[int], count: int) -> np.ndarray:
    classes = np.asarray(forget_classes, dtype=np.int64)
    return np.resize(classes, count)


def generate_class_proxies(
    teacher: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
    cache_path: Path,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    if cache_path.exists():
        with np.load(cache_path, allow_pickle=True) as cached:
            images = cached["images"].astype(np.float32, copy=False)
            labels = cached["labels"].astype(np.int64, copy=False)
            metadata = json.loads(str(cached["metadata"].item()))
        return images, labels, metadata

    set_global_determinism(seed)
    teacher.trainable = False
    all_images: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    target_probabilities: List[float] = []
    remaining = cfg.proxy_count
    target_schedule = balanced_proxy_targets(forget_classes, cfg.proxy_count)
    cursor = 0

    while remaining > 0:
        current = min(cfg.proxy_batch_size, remaining)
        targets = tf.convert_to_tensor(target_schedule[cursor:cursor + current], dtype=tf.int64)
        images = tf.Variable(tf.random.uniform((current,) + bundle.input_shape, 0.0, 255.0, seed=seed + cursor))
        optimizer = tf.keras.optimizers.Adam(cfg.proxy_lr)
        for _ in range(cfg.proxy_steps):
            with tf.GradientTape() as tape:
                output = teacher(images, training=False)
                logits = output_to_logits(output, output_is_logits)
                class_loss = tf.reduce_mean(
                    tf.keras.losses.sparse_categorical_crossentropy(targets, logits, from_logits=True)
                )
                normalized = images / 255.0
                tv_loss = tf.reduce_mean(tf.image.total_variation(normalized))
                l2_loss = tf.reduce_mean(tf.square(normalized - 0.5))
                loss = class_loss + cfg.proxy_tv_weight * tv_loss + cfg.proxy_l2_weight * l2_loss
            gradients = tape.gradient(loss, images)
            optimizer.apply_gradients([(gradients, images)])
            images.assign(tf.clip_by_value(images, 0.0, 255.0))
        probs = output_to_probs(teacher(images, training=False), output_is_logits).numpy()
        target_probabilities.extend(probs[np.arange(current), targets.numpy()].tolist())
        all_images.append(images.numpy().astype(np.float32))
        all_labels.append(targets.numpy())
        cursor += current
        remaining -= current

    proxy_images = np.concatenate(all_images)
    proxy_labels = np.concatenate(all_labels)
    metadata = {
        "mean_target_probability": float(np.mean(target_probabilities)),
        "min_target_probability": float(np.min(target_probabilities)),
        "max_target_probability": float(np.max(target_probabilities)),
        "count": float(len(proxy_images)),
    }
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache_path, images=proxy_images, labels=proxy_labels, metadata=json.dumps(metadata))
    return proxy_images, proxy_labels, metadata


def generate_unsir_error_maximizing_noise(
    teacher: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    count: int,
    steps: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    set_global_determinism(seed)
    targets_np = balanced_proxy_targets(forget_classes, count)
    targets = tf.convert_to_tensor(targets_np, dtype=tf.int64)
    noise = tf.Variable(tf.random.normal((count,) + bundle.input_shape, mean=127.5, stddev=40.0, seed=seed))
    optimizer = tf.keras.optimizers.Adam(1.0)
    teacher.trainable = False
    for _ in range(steps):
        with tf.GradientTape() as tape:
            output = teacher(noise, training=False)
            logits = output_to_logits(output, output_is_logits)
            ce = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(targets, logits, from_logits=True))
            regularizer = 1e-6 * tf.reduce_mean(tf.square(noise - 127.5))
            objective = -ce + regularizer
        gradient = tape.gradient(objective, noise)
        optimizer.apply_gradients([(gradient, noise)])
        noise.assign(tf.clip_by_value(noise, 0.0, 255.0))
    return noise.numpy().astype(np.float32), targets_np

In [ ]:
def categorical_kl(p: tf.Tensor, q: tf.Tensor) -> tf.Tensor:
    p = tf.clip_by_value(p, 1e-7, 1.0)
    q = tf.clip_by_value(q, 1e-7, 1.0)
    return tf.reduce_mean(tf.reduce_sum(p * tf.math.log(p / q), axis=-1))

def scrubref_train(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    proxies: np.ndarray,
    proxy_labels: np.ndarray,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
    update_representation: bool = False,
) -> Tuple[tf.keras.Model, Dict[str, float]]:
    set_global_determinism(seed)
    student = clone_with_weights(original)
    teacher = clone_with_weights(original)
    teacher.trainable = False

    if update_representation:
        set_tail_trainable(student, bundle.num_classes, cfg.representation_tail_layers)
    else:
        set_tail_trainable(student, bundle.num_classes, 0)

    feature_teacher = build_feature_model(teacher, bundle.num_classes)
    feature_student = build_feature_model(student, bundle.num_classes)

    # Precompute centroids for retain representations
    retain_ds_feat = make_dataset(split.retain_train_items[:1000], split.retain_train_labels[:1000], bundle, cfg.eval_batch_size, False, False, False, seed)
    retain_feats_list = []
    for xb, _ in retain_ds_feat:
        retain_feats_list.append(feature_teacher(xb, training=False))
    retain_centroid = tf.reduce_mean(tf.concat(retain_feats_list, axis=0), axis=0)

    optimizer = tf.keras.optimizers.Adam(cfg.scrub_lr)

    retain_ds = make_dataset(split.retain_train_items, split.retain_train_labels, bundle, cfg.batch_size, True, True, True, seed)
    proxy_ds = make_dataset(proxies, proxy_labels, bundle, cfg.proxy_batch_size, True, False, True, seed + 1)

    retain_iter = iter(retain_ds)
    proxy_iter = iter(proxy_ds)

    steps_per_epoch = max(1, len(split.retain_train_items) // cfg.batch_size)

    @tf.function(reduce_retracing=True)
    def train_step(retain_images: tf.Tensor, retain_labels: tf.Tensor, proxy_images: tf.Tensor, proxy_labels: tf.Tensor):
        with tf.GradientTape() as tape:
            # Forward on retain
            student_retain_out = student(retain_images, training=True)
            teacher_retain_out = teacher(retain_images, training=False)

            # Forward on proxy
            student_proxy_out = student(proxy_images, training=True)

            # Loss terms
            ce_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                retain_labels, student_retain_out, from_logits=output_is_logits
            ))

            kd_loss = categorical_kl(
                output_to_probs(teacher_retain_out, output_is_logits, cfg.kd_temperature),
                output_to_probs(student_retain_out, output_is_logits, cfg.kd_temperature)
            )

            forget_loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                proxy_labels, student_proxy_out, from_logits=output_is_logits
            ))

            loss = cfg.retain_ce_weight * ce_loss + cfg.retain_kd_weight * kd_loss + cfg.forget_output_weight * forget_loss

            if update_representation:
                student_retain_feat = feature_student(retain_images, training=True)
                teacher_retain_feat = feature_teacher(retain_images, training=False)
                feat_retain_loss = tf.reduce_mean(tf.square(student_retain_feat - teacher_retain_feat))

                student_proxy_feat = feature_student(proxy_images, training=True)
                feat_forget_loss = tf.reduce_mean(tf.square(student_proxy_feat - retain_centroid))

                loss += cfg.retain_feature_weight * feat_retain_loss + cfg.forget_feature_weight * feat_forget_loss

        gradients = tape.gradient(loss, student.trainable_variables)
        if cfg.gradient_clip_norm > 0:
            gradients, _ = tf.clip_by_global_norm(gradients, cfg.gradient_clip_norm)
        optimizer.apply_gradients(zip(gradients, student.trainable_variables))
        return loss

    # Run scrubbing epochs
    for epoch in range(cfg.scrub_epochs):
        epoch_losses = []
        for _ in range(steps_per_epoch):
            rx, ry = next(retain_iter)
            px, py = next(proxy_iter)
            loss_val = train_step(rx, ry, px, py)
            epoch_losses.append(float(loss_val.numpy()))
        print(f"ScrubRef Epoch {epoch+1}/{cfg.scrub_epochs} Loss: {np.mean(epoch_losses):.4f}")

    return student, {"scrub_final_loss": float(np.mean(epoch_losses))}

## Corrected ScrubRef objectives

The head-only and representation-aware variants use the same output-level objective. The representation-aware variant additionally changes a small tail of the backbone, preserves retained features, and moves proxy features toward the retained feature centroid.

## Baselines

The notebook labels TensorFlow-native adaptations as such. For final publication, report the exact configuration used and verify the adaptation against each official implementation.

In [ ]:
def logit_clamp_baseline(
    original: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    shift: float,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    classifier = classifier_layer(model, bundle.num_classes)
    weights = classifier.get_weights()
    kernel = weights[0].copy()
    kernel[:, list(forget_classes)] = 0.0
    if len(weights) == 2:
        bias = weights[1].copy()
        bias[list(forget_classes)] -= shift
        classifier.set_weights([kernel, bias])
    else:
        classifier.set_weights([kernel])
    return model


def head_reset_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    classifier = classifier_layer(model, bundle.num_classes)
    weights = classifier.get_weights()
    kernel = tf.keras.initializers.GlorotUniform(seed=seed)(weights[0].shape).numpy()
    if len(weights) == 2:
        bias = np.zeros_like(weights[1])
        classifier.set_weights([kernel, bias])
    else:
        classifier.set_weights([kernel])
    return train_supervised(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        cfg.baseline_epochs,
        cfg.batch_size,
        cfg.baseline_lr,
        seed,
        validation=(split.retain_val_items, split.retain_val_labels),
        train_all=False,
        tail_layers=0,
    )


def retain_finetune_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    return train_supervised(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        cfg.baseline_epochs,
        cfg.batch_size,
        cfg.baseline_lr,
        seed,
        validation=(split.retain_val_items, split.retain_val_labels),
        train_all=False,
        tail_layers=cfg.representation_tail_layers,
    )


def unsir_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    noise, noise_labels = generate_unsir_error_maximizing_noise(
        original,
        bundle,
        forget_classes,
        output_is_logits,
        count=min(cfg.proxy_count, 512),
        steps=cfg.unsir_noise_steps,
        seed=seed,
    )
    retain_images, retain_labels = materialize_items(
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        limit=min(len(noise), len(split.retain_train_items)),
        seed=seed,
    )
    impair_images = np.concatenate([retain_images, noise])
    impair_labels = np.concatenate([retain_labels, noise_labels])
    array_bundle = DataBundle(
        name=f"{bundle.name}_materialized",
        train_items=impair_images,
        train_labels=impair_labels,
        test_items=impair_images,
        test_labels=impair_labels,
        class_names=bundle.class_names,
        image_size=bundle.image_size,
        path_mode=False,
    )
    model = train_supervised(
        model,
        impair_images,
        impair_labels,
        array_bundle,
        output_is_logits,
        epochs=1,
        batch_size=cfg.batch_size,
        learning_rate=cfg.unsir_impair_lr,
        seed=seed,
        train_all=True,
    )
    model = train_supervised(
        model,
        retain_images,
        retain_labels,
        array_bundle,
        output_is_logits,
        epochs=max(1, cfg.baseline_epochs // 2),
        batch_size=cfg.batch_size,
        learning_rate=cfg.unsir_repair_lr,
        seed=seed + 1,
        train_all=True,
    )
    return model


def bad_teacher_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    student = clone_with_weights(original)
    competent = clone_with_weights(original)
    incompetent = fresh_clone(original)
    competent.trainable = False
    incompetent.trainable = False
    set_all_trainable(student, True, freeze_batch_norm=True)
    optimizer = tf.keras.optimizers.Adam(cfg.baseline_lr)
    retain_ds = make_dataset(split.retain_train_items, split.retain_train_labels, bundle, cfg.batch_size, True, True, True, seed)
    forget_ds = make_dataset(split.forget_train_items, split.forget_train_labels, bundle, cfg.batch_size, True, True, True, seed + 1)
    retain_it, forget_it = iter(retain_ds), iter(forget_ds)
    steps = max(1, math.ceil(max(len(split.retain_train_items), len(split.forget_train_items)) / cfg.batch_size))

    @tf.function(reduce_retracing=True)
    def step(retain_images: tf.Tensor, retain_labels: tf.Tensor, forget_images: tf.Tensor) -> tf.Tensor:
        with tf.GradientTape() as tape:
            student_retain = student(retain_images, training=True)
            student_forget = student(forget_images, training=True)
            competent_retain = competent(retain_images, training=False)
            incompetent_forget = incompetent(forget_images, training=False)
            ce = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                retain_labels, student_retain, from_logits=output_is_logits
            ))
            kd_retain = categorical_kl(
                output_to_probs(competent_retain, output_is_logits, cfg.kd_temperature),
                output_to_probs(student_retain, output_is_logits, cfg.kd_temperature),
            )
            kd_forget = categorical_kl(
                output_to_probs(incompetent_forget, output_is_logits, cfg.kd_temperature),
                output_to_probs(student_forget, output_is_logits, cfg.kd_temperature),
            )
            loss = ce + kd_retain + kd_forget
        gradients = tape.gradient(loss, student.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, cfg.gradient_clip_norm)
        optimizer.apply_gradients(zip(gradients, student.trainable_variables))
        return loss

    for epoch in range(cfg.baseline_epochs):
        losses = []
        for _ in range(steps):
            retain_images, retain_labels = next(retain_it)
            forget_images, _ = next(forget_it)
            losses.append(float(step(retain_images, retain_labels, forget_images).numpy()))
        print(f"Bad Teacher epoch {epoch + 1}: loss={np.mean(losses):.6f}")
    return student


def scrub_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    student = clone_with_weights(original)
    teacher = clone_with_weights(original)
    teacher.trainable = False
    set_all_trainable(student, True, freeze_batch_norm=True)
    optimizer = tf.keras.optimizers.Adam(cfg.baseline_lr)
    retain_ds = make_dataset(split.retain_train_items, split.retain_train_labels, bundle, cfg.batch_size, True, True, True, seed)
    forget_ds = make_dataset(split.forget_train_items, split.forget_train_labels, bundle, cfg.batch_size, True, True, True, seed + 1)
    retain_it, forget_it = iter(retain_ds), iter(forget_ds)
    steps = max(1, math.ceil(max(len(split.retain_train_items), len(split.forget_train_items)) / cfg.batch_size))

    @tf.function(reduce_retracing=True)
    def retain_step(images: tf.Tensor, labels: tf.Tensor) -> tf.Tensor:
        with tf.GradientTape() as tape:
            student_output = student(images, training=True)
            teacher_output = teacher(images, training=False)
            ce = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                labels, student_output, from_logits=output_is_logits
            ))
            kd = categorical_kl(
                output_to_probs(teacher_output, output_is_logits, cfg.kd_temperature),
                output_to_probs(student_output, output_is_logits, cfg.kd_temperature),
            )
            loss = ce + kd
        gradients = tape.gradient(loss, student.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, cfg.gradient_clip_norm)
        optimizer.apply_gradients(zip(gradients, student.trainable_variables))
        return loss

    @tf.function(reduce_retracing=True)
    def forget_step(images: tf.Tensor) -> tf.Tensor:
        with tf.GradientTape() as tape:
            student_output = student(images, training=True)
            teacher_output = teacher(images, training=False)
            divergence = categorical_kl(
                output_to_probs(teacher_output, output_is_logits, cfg.kd_temperature),
                output_to_probs(student_output, output_is_logits, cfg.kd_temperature),
            )
            loss = -divergence
        gradients = tape.gradient(loss, student.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, cfg.gradient_clip_norm)
        optimizer.apply_gradients(zip(gradients, student.trainable_variables))
        return divergence

    for epoch in range(cfg.baseline_epochs):
        retain_losses = []
        forget_divergences = []
        for _ in range(steps):
            forget_images, _ = next(forget_it)
            forget_divergences.append(float(forget_step(forget_images).numpy()))
            retain_images, retain_labels = next(retain_it)
            retain_losses.append(float(retain_step(retain_images, retain_labels).numpy()))
        print(
            f"SCRUB epoch {epoch + 1}: retain_loss={np.mean(retain_losses):.6f}, "
            f"forget_divergence={np.mean(forget_divergences):.6f}"
        )
    return student


def estimate_fisher_diagonal(
    model: tf.keras.Model,
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    output_is_logits: bool,
    batch_size: int,
    max_batches: int,
    seed: int,
) -> List[tf.Tensor]:
    set_all_trainable(model, True, freeze_batch_norm=True)
    variables = model.trainable_variables
    fisher = [tf.zeros_like(variable, dtype=tf.float32) for variable in variables]
    ds = make_dataset(items, labels, bundle, batch_size, True, False, False, seed)
    batch_count = 0
    for images, targets in ds.take(max_batches):
        with tf.GradientTape() as tape:
            output = model(images, training=False)
            loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                targets, output, from_logits=output_is_logits
            ))
        gradients = tape.gradient(loss, variables)
        fisher = [
            accumulated + (tf.square(tf.cast(gradient, tf.float32)) if gradient is not None else 0.0)
            for accumulated, gradient in zip(fisher, gradients)
        ]
        batch_count += 1
    return [value / max(batch_count, 1) for value in fisher]


def ssd_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    retain_fisher = estimate_fisher_diagonal(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        cfg.batch_size,
        cfg.ssd_fisher_batches,
        seed,
    )
    forget_fisher = estimate_fisher_diagonal(
        model,
        split.forget_train_items,
        split.forget_train_labels,
        bundle,
        output_is_logits,
        cfg.batch_size,
        cfg.ssd_fisher_batches,
        seed + 1,
    )
    epsilon = 1e-12
    for variable, retain_value, forget_value in zip(model.trainable_variables, retain_fisher, forget_fisher):
        ratio = forget_value / (retain_value + epsilon)
        selected = ratio > cfg.ssd_selection_weighting
        dampening = tf.minimum(
            cfg.ssd_dampening_constant * retain_value / (forget_value + epsilon),
            tf.ones_like(forget_value),
        )
        factor = tf.where(selected, dampening, tf.ones_like(dampening))
        variable.assign(tf.cast(variable, tf.float32) * factor)
    return model


def retrain_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    initial_path = cfg.initial_weights_paths.get(cfg.dataset, "")
    model = fresh_clone(original, initial_path)
    return train_supervised(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        cfg.retrain_epochs,
        cfg.batch_size,
        cfg.retrain_lr,
        seed,
        validation=(split.retain_val_items, split.retain_val_labels),
        train_all=True,
    )

## Experiment runner

Each method is saved and evaluated immediately so Colab does not keep every model in GPU or CPU memory. The runner clears TensorFlow and Python allocations between methods and seeds. Existing model files are loaded unless `force_rerun=True`, so interrupted publication runs can resume.


In [ ]:
def method_model_path(run_dir: Path, method: str) -> Path:
    return run_dir / "models" / f"{method}.keras"


def save_json(path: Path, data: Dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, default=str), encoding="utf-8")


def build_method(
    method: str,
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    proxies: np.ndarray,
    proxy_labels: np.ndarray,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> Tuple[tf.keras.Model, Dict[str, float]]:
    metadata: Dict[str, float] = {}
    if method == "original":
        return clone_with_weights(original), metadata
    if method == "retrained":
        return retrain_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "logit_clamp":
        return logit_clamp_baseline(original, bundle, forget_classes, shift=20.0), metadata
    if method == "head_reset":
        return head_reset_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "retain_finetune":
        return retain_finetune_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "scrubref_head":
        return scrubref_train(
            original, split, bundle, proxies, proxy_labels, forget_classes, output_is_logits, cfg, seed, False
        )
    if method == "scrubref_rep":
        return scrubref_train(
            original, split, bundle, proxies, proxy_labels, forget_classes, output_is_logits, cfg, seed, True
        )
    if method == "unsir":
        return unsir_baseline(original, split, bundle, forget_classes, output_is_logits, cfg, seed), metadata
    if method == "bad_teacher":
        return bad_teacher_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "scrub":
        return scrub_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "ssd":
        return ssd_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    raise ValueError(f"Unknown method: {method}")


def load_saved_model(path: Path) -> tf.keras.Model:
    return tf.keras.models.load_model(path, custom_objects=custom_objects(), compile=False, safe_mode=False)


def run_seed(
    cfg: ExperimentConfig,
    seed: int,
    force_rerun: bool = False,
) -> Dict[str, Path]:
    reset_tensorflow(seed)
    bundle = load_dataset(cfg)
    print(
        "Dataset storage:",
        bundle.train_items.dtype,
        f"{array_memory_mb(bundle.train_items, bundle.test_items):.2f} MB",
    )
    if not bundle.path_mode and bundle.train_items.dtype != np.uint8:
        raise RuntimeError(
            "CIFAR arrays must remain uint8 in RAM. Re-run the RAM-safe dataset cell."
        )
    forget_classes = cfg.forget_classes_by_dataset[cfg.dataset]
    split = make_unlearning_split(bundle, forget_classes, cfg.retrain_validation_fraction, seed)
    print(
        "Bundle + split image arrays:",
        f"{array_memory_mb(bundle.train_items, bundle.test_items, split.retain_train_items, split.retain_val_items, split.forget_train_items, split.retain_test_items, split.forget_test_items):.2f} MB",
    )

    run_dir = Path(cfg.results_root) / cfg.dataset / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "models").mkdir(parents=True, exist_ok=True)
    (run_dir / "figures").mkdir(parents=True, exist_ok=True)
    (run_dir / "features").mkdir(parents=True, exist_ok=True)
    save_json(run_dir / "config.json", asdict(cfg))

    original, output_is_logits, original_source = load_or_train_original(cfg, bundle, seed, run_dir)
    original_run_path = method_model_path(run_dir, "original")
    if not original_run_path.exists() or force_rerun:
        original.save(original_run_path)

    proxy_path = run_dir / "proxies.npz"
    proxies, proxy_labels, proxy_metadata = generate_class_proxies(
        original,
        bundle,
        forget_classes,
        output_is_logits,
        cfg,
        seed,
        proxy_path,
    )
    save_json(run_dir / "proxy_metadata.json", proxy_metadata)

    utility_rows: List[Dict[str, object]] = []
    mia_rows: List[Dict[str, object]] = []
    runtime_rows: List[Dict[str, object]] = []
    per_class_rows: List[Dict[str, object]] = []

    for method in cfg.enabled_methods:
        reset_tensorflow(seed)
        original = load_saved_model(original_run_path)
        model_path = method_model_path(run_dir, method)
        method_metadata_path = run_dir / "models" / f"{method}_metadata.json"
        try:
            tf.config.experimental.reset_memory_stats("GPU:0")
        except Exception:
            pass
        start = time.perf_counter()
        metadata: Dict[str, float] = {}
        if model_path.exists() and not force_rerun:
            model = load_saved_model(model_path)
            reused = 1.0
            if method_metadata_path.exists():
                stored_metadata = json.loads(method_metadata_path.read_text(encoding="utf-8"))
                metadata = {
                    key: float(value) for key, value in stored_metadata.get("method_metadata", {}).items()
                    if isinstance(value, (int, float))
                }
                runtime_seconds = float(stored_metadata.get("runtime_seconds", float("nan")))
            else:
                runtime_seconds = float("nan")
        else:
            model, metadata = build_method(
                method,
                original,
                split,
                bundle,
                proxies,
                proxy_labels,
                forget_classes,
                output_is_logits,
                cfg,
                seed,
            )
            reused = 0.0
            runtime_seconds = time.perf_counter() - start
            if cfg.save_models:
                model.save(model_path)
            save_json(method_metadata_path, {
                "runtime_seconds": runtime_seconds,
                "method_metadata": metadata,
            })

        utility, class_accuracy = evaluate_utility(
            model, bundle, forget_classes, output_is_logits, cfg.eval_batch_size, seed
        )
        mia = evaluate_membership_inference(
            model,
            split,
            bundle,
            output_is_logits,
            cfg.eval_batch_size,
            cfg.mia_attack_repeats,
            seed,
        )
        parameter_stats = parameter_change_statistics(original, model)

        utility_rows.append({"seed": seed, "method": method, **utility, **metadata})
        mia_rows.append({"seed": seed, "method": method, **mia})
        runtime_rows.append({
            "seed": seed,
            "method": method,
            "runtime_seconds": runtime_seconds,
            "peak_gpu_memory_mb": peak_gpu_memory_mb(),
            "reused_saved_model": reused,
            "original_source": original_source,
            "output_is_logits": float(output_is_logits),
            **parameter_stats,
        })
        for class_id, accuracy in enumerate(class_accuracy):
            per_class_rows.append({
                "seed": seed,
                "method": method,
                "class_id": class_id,
                "class_name": bundle.class_names[class_id],
                "is_forgotten": int(class_id in forget_classes),
                "accuracy": float(accuracy),
            })
        print("Completed", method, utility)
        del model, original
        release_memory(clear_session=True)

    pd.DataFrame(utility_rows).to_csv(run_dir / "utility_metrics.csv", index=False)
    pd.DataFrame(mia_rows).to_csv(run_dir / "mia_metrics.csv", index=False)
    pd.DataFrame(runtime_rows).to_csv(run_dir / "runtime_metrics.csv", index=False)
    pd.DataFrame(per_class_rows).to_csv(run_dir / "per_class_accuracy.csv", index=False)

    run_representation_audit(cfg, run_dir, bundle, split, forget_classes, output_is_logits, seed)
    make_seed_figures(run_dir)
    outputs = {
        "run_dir": run_dir,
        "utility": run_dir / "utility_metrics.csv",
        "mia": run_dir / "mia_metrics.csv",
        "runtime": run_dir / "runtime_metrics.csv",
        "representation": run_dir / "representation_metrics.csv",
    }
    del proxies, proxy_labels, split, bundle
    release_memory(clear_session=True)
    return outputs

In [ ]:
def run_representation_audit(
    cfg: ExperimentConfig,
    run_dir: Path,
    bundle: DataBundle,
    split: UnlearningSplit,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    seed: int,
) -> None:
    train_idx = stratified_indices(bundle.train_labels, cfg.probe_train_limit, seed)
    test_idx = stratified_indices(bundle.test_labels, cfg.probe_test_limit, seed + 1)
    cka_idx = stratified_indices(bundle.test_labels, cfg.cka_sample_limit, seed + 2)

    feature_paths: Dict[str, Path] = {}
    probability_paths: Dict[str, Path] = {}
    for method in cfg.representation_methods:
        path = method_model_path(run_dir, method)
        if not path.exists():
            continue
        reset_tensorflow(seed)
        model = load_saved_model(path)
        train_features, train_labels = extract_features(
            model,
            bundle.train_items[train_idx],
            bundle.train_labels[train_idx],
            bundle,
            cfg.eval_batch_size,
            seed,
        )
        test_features, test_labels = extract_features(
            model,
            bundle.test_items[test_idx],
            bundle.test_labels[test_idx],
            bundle,
            cfg.eval_batch_size,
            seed,
        )
        cka_features, cka_labels = extract_features(
            model,
            bundle.test_items[cka_idx],
            bundle.test_labels[cka_idx],
            bundle,
            cfg.eval_batch_size,
            seed,
        )
        probs, prob_labels = predict_probabilities(
            model,
            bundle.test_items[cka_idx],
            bundle.test_labels[cka_idx],
            bundle,
            output_is_logits,
            cfg.eval_batch_size,
            seed,
        )
        feature_path = run_dir / "features" / f"{method}.npz"
        probability_path = run_dir / "features" / f"{method}_probs.npz"
        np.savez_compressed(
            feature_path,
            train_features=train_features.astype(np.float32, copy=False),
            train_labels=train_labels,
            test_features=test_features.astype(np.float32, copy=False),
            test_labels=test_labels,
            cka_features=cka_features.astype(np.float32, copy=False),
            cka_labels=cka_labels,
        )
        np.savez_compressed(
            probability_path,
            probabilities=probs.astype(np.float32, copy=False),
            labels=prob_labels,
        )
        feature_paths[method] = feature_path
        probability_paths[method] = probability_path
        del (
            model,
            train_features,
            train_labels,
            test_features,
            test_labels,
            cka_features,
            cka_labels,
            probs,
            prob_labels,
        )
        release_memory(clear_session=True)

    if "original" not in feature_paths or "original" not in probability_paths:
        raise RuntimeError("Representation audit requires the saved original model.")

    with np.load(feature_paths["original"]) as data:
        original_features = data["cka_features"].astype(np.float32, copy=True)
    retrained_features = None
    retrained_probs = None
    if "retrained" in feature_paths:
        with np.load(feature_paths["retrained"]) as data:
            retrained_features = data["cka_features"].astype(np.float32, copy=True)
    if "retrained" in probability_paths:
        with np.load(probability_paths["retrained"]) as data:
            retrained_probs = data["probabilities"].astype(np.float32, copy=True)
    with np.load(probability_paths["original"]) as data:
        labels = data["labels"].astype(np.int64, copy=True)

    forget_mask = np.isin(labels, np.asarray(forget_classes))
    retain_mask = ~forget_mask

    rows: List[Dict[str, object]] = []
    for method, feature_path in feature_paths.items():
        with np.load(feature_path) as data:
            train_features = data["train_features"].astype(np.float32, copy=True)
            train_labels = data["train_labels"].astype(np.int64, copy=True)
            test_features = data["test_features"].astype(np.float32, copy=True)
            test_labels = data["test_labels"].astype(np.int64, copy=True)
            method_features = data["cka_features"].astype(np.float32, copy=True)

        probe = representation_probe_metrics(
            train_features,
            train_labels,
            test_features,
            test_labels,
            forget_classes,
        )
        row: Dict[str, object] = {
            "seed": seed,
            "method": method,
            "cka_to_original": linear_cka(method_features, original_features),
            **probe,
        }
        if retrained_features is not None:
            row["cka_to_retrained"] = linear_cka(method_features, retrained_features)
        if retrained_probs is not None and method in probability_paths:
            with np.load(probability_paths[method]) as data:
                method_probs = data["probabilities"].astype(np.float32, copy=True)
            row["js_to_retrained_all"] = jensen_shannon_divergence(method_probs, retrained_probs)
            row["js_to_retrained_retain"] = jensen_shannon_divergence(
                method_probs[retain_mask], retrained_probs[retain_mask]
            )
            row["js_to_retrained_forget"] = jensen_shannon_divergence(
                method_probs[forget_mask], retrained_probs[forget_mask]
            )
            row["prediction_disagreement_to_retrained"] = float(
                np.mean(method_probs.argmax(axis=1) != retrained_probs.argmax(axis=1))
            )
            del method_probs
        rows.append(row)
        del train_features, train_labels, test_features, test_labels, method_features, probe
        release_memory(clear_session=False)

    pd.DataFrame(rows).to_csv(run_dir / "representation_metrics.csv", index=False)
    del original_features, retrained_features, retrained_probs, labels, train_idx, test_idx, cka_idx
    release_memory(clear_session=True)


def make_seed_figures(run_dir: Path) -> None:
    utility = pd.read_csv(run_dir / "utility_metrics.csv")
    mia = pd.read_csv(run_dir / "mia_metrics.csv")
    representation_path = run_dir / "representation_metrics.csv"
    representation = pd.read_csv(representation_path) if representation_path.exists() else pd.DataFrame()

    figure_dir = run_dir / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)

    positions = np.arange(len(utility))
    width = 0.36
    fig, ax = plt.subplots(figsize=(max(10, len(utility) * 1.1), 5.5))
    ax.bar(positions - width / 2, utility["retained_accuracy"], width, label="Retained accuracy")
    ax.bar(positions + width / 2, utility["forgotten_accuracy"], width, label="Forgotten accuracy")
    ax.set_xticks(positions, utility["method"], rotation=35, ha="right")
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Accuracy")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "retained_forgotten_accuracy.png", dpi=220)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(max(10, len(mia) * 1.1), 5.5))
    ax.bar(positions - width / 2, mia["mia_confidence_auc"], width, label="Confidence MIA")
    ax.bar(positions + width / 2, mia["mia_learned_auc"], width, label="Learned MIA")
    ax.axhline(0.5, linestyle="--", linewidth=1.0)
    ax.set_xticks(positions, mia["method"], rotation=35, ha="right")
    ax.set_ylim(0.45, 1.0)
    ax.set_ylabel("ROC-AUC")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "membership_inference_auc.png", dpi=220)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(max(10, len(utility) * 1.1), 5.5))
    ax.bar(positions - width / 2, utility["ece_15"], width, label="ECE")
    ax.bar(positions + width / 2, utility["brier"], width, label="Brier")
    ax.set_xticks(positions, utility["method"], rotation=35, ha="right")
    ax.set_ylabel("Calibration error")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "calibration_metrics.png", dpi=220)
    plt.close(fig)

    if not representation.empty and "cka_to_retrained" in representation.columns:
        positions_rep = np.arange(len(representation))
        fig, ax = plt.subplots(figsize=(max(10, len(representation) * 1.1), 5.5))
        ax.bar(positions_rep - width / 2, representation["cka_to_original"], width, label="CKA to original")
        ax.bar(positions_rep + width / 2, representation["cka_to_retrained"], width, label="CKA to retrained")
        ax.set_xticks(positions_rep, representation["method"], rotation=35, ha="right")
        ax.set_ylim(0.0, 1.0)
        ax.set_ylabel("Linear CKA")
        ax.legend()
        fig.tight_layout()
        fig.savefig(figure_dir / "representation_cka.png", dpi=220)
        plt.close(fig)

## Execute experiments

The publication configuration runs all seeds. The runner resumes from saved `.keras` files. Set `FORCE_RERUN=True` only when code or hyperparameters change.

## RAM preflight

This preflight confirms that CIFAR data remain `uint8`. It does not retain the dataset after the check. A CIFAR-100 bundle should occupy roughly 176 MB before the split.


In [ ]:
PREFLIGHT_BUNDLE = load_dataset(CFG)
print("Train dtype:", PREFLIGHT_BUNDLE.train_items.dtype)
print("Test dtype:", PREFLIGHT_BUNDLE.test_items.dtype)
print(
    "Dataset image RAM:",
    f"{array_memory_mb(PREFLIGHT_BUNDLE.train_items, PREFLIGHT_BUNDLE.test_items):.2f} MB",
)
if not PREFLIGHT_BUNDLE.path_mode and PREFLIGHT_BUNDLE.train_items.dtype != np.uint8:
    raise RuntimeError("RAM preflight failed: CIFAR images are not uint8.")
del PREFLIGHT_BUNDLE
release_memory(clear_session=False)
print("RAM preflight passed.")


/tmp/ipykernel_5410/2422473100.py:52: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(root_path)


Train dtype: <U64
Test dtype: <U62
Dataset image RAM: 3.13 MB
RAM preflight passed.


In [ ]:
#import os
#import shutil

#run_dir = Path(cfg.results_root) / cfg.dataset / f"seed_{seed}"
#models_dir = os.path.join(run_dir, "models")
#os.makedirs(models_dir, exist_ok=True)

#src_trained = os.path.join(run_dir, "trained_original.keras")
#src_original = os.path.join(run_dir, "original.keras")
#dst_models = os.path.join(models_dir, "original.keras")

#if os.path.exists(src_trained) and not os.path.exists(dst_models):
#    shutil.copy(src_trained, dst_models)
#    print("The original model was successfully moved to the models folder.")
#elif os.path.exists(src_original) and not os.path.exists(dst_models):
#    shutil.copy(src_original, dst_models)
#    print("The original model was moved to the models folder.")

#print("\nContents of the models folder:")
#print(os.listdir(models_dir))


📁 محتویات پوشه models:
[]


In [ ]:
import inspect
print(inspect.getsource(load_or_train_original))

def load_or_train_original(
    cfg: ExperimentConfig,
    bundle: DataBundle,
    seed: int,
    run_dir: Path,
) -> Tuple[tf.keras.Model, bool, str]:
    checkpoint = Path(cfg.checkpoint_paths[cfg.dataset])
    if checkpoint.exists():
        model = tf.keras.models.load_model(checkpoint, custom_objects=custom_objects(), compile=False, safe_mode=False)
        source = str(checkpoint)
    else:
        model = build_resnet18(bundle.input_shape, bundle.num_classes)
        sample_ds = make_dataset(bundle.train_items[:16], bundle.train_labels[:16], bundle, 16, False, False, False, seed)
        sample_x, _ = next(iter(sample_ds))
        output_is_logits = infer_output_is_logits(model, sample_x, True)
        model = train_supervised(
            model,
            bundle.train_items,
            bundle.train_labels,
            bundle,
            output_is_logits,
            cfg.original_epochs,
            cfg.batch_size,
            cfg.original_lr,
            seed,
            v

In [ ]:
CFG.enabled_methods = [
    "original", "retrained", "logit_clamp",
    "head_reset", "retain_finetune"
    # "scrubref_head"  <-- این بخش را موقتا کامنت یا حذف کنید
]

In [ ]:
import os
import pandas as pd
csv_path = Path(CFG.results_root) / CFG.dataset / f"seed_{42}" / "utility_metrics.csv"

#csv_path = "/content/drive/MyDrive/scrubref_revision_results/cifar100/seed_42/utility_metrics.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    cols = ["method", "overall_accuracy", "retained_accuracy", "forgotten_accuracy", "macro_f1", "nll"]
    available_cols = [c for c in cols if c in df.columns]
    display(df[available_cols])
else:
    print(f"⚠️ Result file not found at: {csv_path}")
    print("Please execute the experiment runner cell first (under 'Execute experiments') to generate the results.")

⚠️ Result file not found at: results/imagewoof/seed_42/utility_metrics.csv
Please execute the experiment runner cell first (under 'Execute experiments') to generate the results.


In [ ]:
# Missing representation audit helper definitions
def stratified_indices(labels: np.ndarray, limit: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(labels)
    samples_per_class = max(1, limit // len(unique_classes))
    selected_indices = []
    for class_id in unique_classes:
        indices = np.flatnonzero(labels == class_id)
        if len(indices) > 0:
            chosen = rng.choice(indices, min(len(indices), samples_per_class), replace=False)
            selected_indices.extend(chosen)
    return np.array(selected_indices, dtype=np.int64)

def extract_features(model: tf.keras.Model, items: np.ndarray, labels: np.ndarray, bundle: DataBundle, batch_size: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    feature_model = build_feature_model(model, bundle.num_classes)
    ds = make_dataset(items, labels, bundle, batch_size, False, False, False, seed)
    features = []
    labels_out = []
    for xb, yb in ds:
        features.append(feature_model(xb, training=False).numpy())
        labels_out.append(yb.numpy())
    return np.concatenate(features), np.concatenate(labels_out)

def linear_cka(X: np.ndarray, Y: np.ndarray) -> float:
    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)
    XTX = np.dot(X, X.T)
    YTY = np.dot(Y, Y.T)
    num = np.sum(XTX * YTY)
    den = np.sqrt(np.sum(XTX * XTX) * np.sum(YTY * YTY))
    return float(num / max(den, 1e-12))

def jensen_shannon_divergence(P: np.ndarray, Q: np.ndarray) -> float:
    P = np.clip(P, 1e-7, 1.0)
    Q = np.clip(Q, 1e-7, 1.0)
    M = 0.5 * (P + Q)
    kl_pm = np.sum(P * np.log(P / M), axis=1)
    kl_qm = np.sum(Q * np.log(Q / M), axis=1)
    return float(np.mean(0.5 * (kl_pm + kl_qm)))

def representation_probe_metrics(train_features: np.ndarray, train_labels: np.ndarray, test_features: np.ndarray, test_labels: np.ndarray, forget_classes: Sequence[int]) -> Dict[str, float]:
    # Simple linear probe audit implementation
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(train_features)
    X_te = scaler.transform(test_features)

    probe = LogisticRegression(max_iter=150, class_weight='balanced', C=0.1)
    probe.fit(X_tr, train_labels)
    preds = probe.predict(X_te)

    forget_mask = np.isin(test_labels, np.asarray(forget_classes))

    return {
        "linear_probe_overall_accuracy": float(accuracy_score(test_labels, preds)),
        "linear_probe_retained_accuracy": float(accuracy_score(test_labels[~forget_mask], preds[~forget_mask])),
        "linear_probe_forgotten_accuracy": float(accuracy_score(test_labels[forget_mask], preds[forget_mask]))
    }


In [ ]:
# --------------------------------------------------------------------------------
# FORCE_RERUN Configuration Guide:
#
# False (Default):
# If the experiment pipeline was already executed and saved models (.keras) exist
# in the results directory, it will load the cached weights. This saves substantial
# runtime by skipping redundant training (especially the 100-epoch base model training).
#
# True:
# Set to True if you made changes to the code, network architecture, unlearning
# hyperparameters, or loss formulations, and want to force a fresh re-run of
# all training and evaluation routines from scratch.
# --------------------------------------------------------------------------------
FORCE_RERUN = False

SEED_OUTPUTS = []
for experiment_seed in CFG.seeds:
    print(f"\n{'=' * 90}\nRunning {CFG.dataset}, seed={experiment_seed}\n{'=' * 90}")
    SEED_OUTPUTS.append(run_seed(CFG, experiment_seed, force_rerun=FORCE_RERUN))

print("\n" + "="*90 + "\nAll experiments completed successfully!\n" + "="*90)
SEED_OUTPUTS

## Aggregate all seeds

The aggregation cell produces mean, standard deviation, and 95% normal-approximation confidence intervals for every numeric metric.

In [ ]:
def aggregate_metric_file(cfg: ExperimentConfig, filename: str, output_name: str) -> pd.DataFrame:
    frames = []
    for seed in cfg.seeds:
        path = Path(cfg.results_root) / cfg.dataset / f"seed_{seed}" / filename
        if path.exists():
            frames.append(pd.read_csv(path))
    if not frames:
        raise FileNotFoundError(f"No {filename} files were found.")
    combined = pd.concat(frames, ignore_index=True)
    numeric_columns = [column for column in combined.select_dtypes(include=[np.number]).columns if column != "seed"]
    grouped = combined.groupby("method")[numeric_columns].agg(["mean", "std", "count"])
    rows = []
    for method in grouped.index:
        row: Dict[str, object] = {"method": method}
        for metric in numeric_columns:
            mean = grouped.loc[method, (metric, "mean")]
            std = grouped.loc[method, (metric, "std")]
            count = grouped.loc[method, (metric, "count")]
            ci95 = 1.96 * std / math.sqrt(count) if count > 1 and np.isfinite(std) else float("nan")
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
            row[f"{metric}_ci95"] = ci95
        rows.append(row)
    output = pd.DataFrame(rows)
    aggregate_dir = Path(cfg.results_root) / cfg.dataset / "aggregate"
    aggregate_dir.mkdir(parents=True, exist_ok=True)
    combined.to_csv(aggregate_dir / f"all_seeds_{filename}", index=False)
    output.to_csv(aggregate_dir / output_name, index=False)
    return output


AGGREGATED_UTILITY = aggregate_metric_file(CFG, "utility_metrics.csv", "utility_summary.csv")
AGGREGATED_MIA = aggregate_metric_file(CFG, "mia_metrics.csv", "mia_summary.csv")
AGGREGATED_RUNTIME = aggregate_metric_file(CFG, "runtime_metrics.csv", "runtime_summary.csv")
AGGREGATED_REPRESENTATION = aggregate_metric_file(CFG, "representation_metrics.csv", "representation_summary.csv")

print("Utility summary")
display(AGGREGATED_UTILITY)
print("MIA summary")
display(AGGREGATED_MIA)
print("Representation summary")
display(AGGREGATED_REPRESENTATION)

Utility summary


,method,overall_accuracy_mean,overall_accuracy_std,overall_accuracy_ci95,retained_accuracy_mean,retained_accuracy_std,retained_accuracy_ci95,forgotten_accuracy_mean,forgotten_accuracy_std,forgotten_accuracy_ci95,...,brier_ci95,ece_15_mean,ece_15_std,ece_15_ci95,mean_global_forget_mass_mean,mean_global_forget_mass_std,mean_global_forget_mass_ci95,mean_forget_test_true_class_confidence_mean,mean_forget_test_true_class_confidence_std,mean_forget_test_true_class_confidence_ci95
0,head_reset,0.137185,NaN,NaN,0.153125,NaN,NaN,0.0,NaN,NaN,...,NaN,0.021339,NaN,NaN,1.023739e-03,NaN,NaN,1.033014e-03,NaN,NaN
1,logit_clamp,0.142275,NaN,NaN,0.158807,NaN,NaN,0.0,NaN,NaN,...,NaN,0.039513,NaN,NaN,1.998138e-10,NaN,NaN,2.024438e-10,NaN,NaN
2,original,0.142275,NaN,NaN,0.158807,NaN,NaN,0.0,NaN,NaN,...,NaN,0.030833,NaN,NaN,4.809709e-02,NaN,NaN,5.081105e-02,NaN,NaN
3,retain_finetune,0.260626,NaN,NaN,0.290909,NaN,NaN,0.0,NaN,NaN,...,NaN,0.079186,NaN,NaN,1.035165e-04,NaN,NaN,1.463866e-04,NaN,NaN
4,retrained,0.223976,NaN,NaN,0.250000,NaN,NaN,0.0,NaN,NaN,...,NaN,0.374043,NaN,NaN,7.730919e-06,NaN,NaN,7.398756e-06,NaN,NaN


MIA summary


,method,mia_confidence_auc_mean,mia_confidence_auc_std,mia_confidence_auc_ci95,mia_loss_auc_mean,mia_loss_auc_std,mia_loss_auc_ci95,mia_entropy_auc_mean,mia_entropy_auc_std,mia_entropy_auc_ci95,...,mia_modified_entropy_auc_ci95,mia_learned_auc_mean,mia_learned_auc_std,mia_learned_auc_ci95,mia_learned_ap_mean,mia_learned_ap_std,mia_learned_ap_ci95,mia_member_count_mean,mia_member_count_std,mia_member_count_ci95
0,head_reset,0.513358,NaN,NaN,0.513358,NaN,NaN,0.541188,NaN,NaN,...,NaN,0.511387,NaN,NaN,0.512251,NaN,NaN,409.0,NaN,NaN
1,logit_clamp,0.516305,NaN,NaN,0.500000,NaN,NaN,0.506115,NaN,NaN,...,NaN,0.471674,NaN,NaN,0.502438,NaN,NaN,409.0,NaN,NaN
2,original,0.531372,NaN,NaN,0.531372,NaN,NaN,0.512347,NaN,NaN,...,NaN,0.480139,NaN,NaN,0.511208,NaN,NaN,409.0,NaN,NaN
3,retain_finetune,0.544150,NaN,NaN,0.544150,NaN,NaN,0.518630,NaN,NaN,...,NaN,0.514156,NaN,NaN,0.522103,NaN,NaN,409.0,NaN,NaN
4,retrained,0.518098,NaN,NaN,0.518116,NaN,NaN,0.508838,NaN,NaN,...,NaN,0.490493,NaN,NaN,0.498754,NaN,NaN,409.0,NaN,NaN


Representation summary


,method,cka_to_original_mean,cka_to_original_std,cka_to_original_ci95,linear_probe_overall_accuracy_mean,linear_probe_overall_accuracy_std,linear_probe_overall_accuracy_ci95,linear_probe_retained_accuracy_mean,linear_probe_retained_accuracy_std,linear_probe_retained_accuracy_ci95,...,js_to_retrained_all_ci95,js_to_retrained_retain_mean,js_to_retrained_retain_std,js_to_retrained_retain_ci95,js_to_retrained_forget_mean,js_to_retrained_forget_std,js_to_retrained_forget_ci95,prediction_disagreement_to_retrained_mean,prediction_disagreement_to_retrained_std,prediction_disagreement_to_retrained_ci95
0,logit_clamp,1.000000,NaN,NaN,0.2935,NaN,NaN,0.302222,NaN,NaN,...,NaN,0.250128,NaN,NaN,0.224702,NaN,NaN,0.886667,NaN,NaN
1,original,1.000000,NaN,NaN,0.2935,NaN,NaN,0.302222,NaN,NaN,...,NaN,0.261759,NaN,NaN,0.238355,NaN,NaN,0.886667,NaN,NaN
2,retrained,0.105389,NaN,NaN,0.3610,NaN,NaN,0.367222,NaN,NaN,...,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN


## Reviewer-focused result checks

These checks do not decide acceptance. They identify invalid or weak result patterns before values are copied into the manuscript.

In [ ]:
def reviewer_readiness_checks(cfg: ExperimentConfig) -> pd.DataFrame:
    aggregate_dir = Path(cfg.results_root) / cfg.dataset / "aggregate"
    utility = pd.read_csv(aggregate_dir / "utility_summary.csv").set_index("method")
    mia = pd.read_csv(aggregate_dir / "mia_summary.csv").set_index("method")
    representation = pd.read_csv(aggregate_dir / "representation_summary.csv").set_index("method")
    rows = []
    for method in ["scrubref_head", "scrubref_rep"]:
        if method not in utility.index:
            continue
        retain_drop = utility.loc["original", "retained_accuracy_mean"] - utility.loc[method, "retained_accuracy_mean"]
        forgotten_accuracy = utility.loc[method, "forgotten_accuracy_mean"]
        learned_mia = mia.loc[method, "mia_learned_auc_mean"]
        cka_original = representation.loc[method, "cka_to_original_mean"]
        cka_retrained = representation.loc[method, "cka_to_retrained_mean"]
        linear_probe_forget = representation.loc[method, "linear_probe_forgotten_accuracy_mean"]
        rows.append({
            "method": method,
            "retained_accuracy_drop": retain_drop,
            "forgotten_accuracy": forgotten_accuracy,
            "learned_mia_auc": learned_mia,
            "cka_to_original": cka_original,
            "cka_to_retrained": cka_retrained,
            "closer_to_retrained_than_original": bool(cka_retrained > cka_original),
            "linear_probe_forgotten_accuracy": linear_probe_forget,
            "output_forgetting_pass": bool(forgotten_accuracy <= 0.05),
            "empirical_mia_pass": bool(learned_mia <= 0.60),
            "utility_pass": bool(retain_drop <= 0.03),
        })
    checks = pd.DataFrame(rows)
    checks.to_csv(aggregate_dir / "reviewer_readiness_checks.csv", index=False)
    return checks


REVIEWER_CHECKS = reviewer_readiness_checks(CFG)
display(REVIEWER_CHECKS)

""


## Optional ablation runner

The main result must be compared with mechanisms that can trivially suppress outputs. This ablation executes the two ScrubRef variants under four settings:

- no logit surgery, no clamp;
- logit surgery only;
- clamp only;
- both mechanisms.

In [ ]:
def run_scrubref_ablations(base_cfg: ExperimentConfig, seed: int) -> pd.DataFrame:
    rows = []
    settings = [
        (False, False, "none"),
        (True, False, "logit_surgery"),
        (False, True, "confidence_clamp"),
        (True, True, "surgery_and_clamp"),
    ]
    for use_surgery, use_clamp, label in settings:
        ablation_cfg = ExperimentConfig(**asdict(base_cfg))
        ablation_cfg.seeds = (seed,)
        ablation_cfg.enabled_methods = ("scrubref_head", "scrubref_rep")
        ablation_cfg.representation_methods = ("original", "retrained", "scrubref_head", "scrubref_rep")
        ablation_cfg.enable_main_method_logit_surgery = use_surgery
        ablation_cfg.enable_main_method_clamp = use_clamp
        ablation_cfg.results_root = str(Path(base_cfg.results_root) / "ablations" / label)
        outputs = run_seed(ablation_cfg, seed, force_rerun=False)
        utility = pd.read_csv(outputs["utility"])
        utility["ablation"] = label
        rows.append(utility)
    result = pd.concat(rows, ignore_index=True)
    output_dir = Path(base_cfg.results_root) / "ablations"
    output_dir.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_dir / f"{base_cfg.dataset}_ablation_results.csv", index=False)
    return result


RUN_ABLATIONS = True
ABLATION_RESULTS = run_scrubref_ablations(CFG, CFG.seeds[0]) if RUN_ABLATIONS else pd.DataFrame()
display(ABLATION_RESULTS)

KeyboardInterrupt: 

## Output package

After completion, the aggregate directory contains:

- `utility_summary.csv`
- `mia_summary.csv`
- `runtime_summary.csv`
- `representation_summary.csv`
- `reviewer_readiness_checks.csv`
- all per-seed result files, model checkpoints, proxy arrays, and figures

Use only values generated by these files in the revised manuscript. Do not reuse numerical results from the submitted notebook because its scrub branch and membership evaluation were invalid.

## Method-label discipline

- `scrubref_head` is the corrected original algorithmic scope.
- `scrubref_rep` is an algorithmic extension and must be described as such.
- `unsir`, `bad_teacher`, `scrub`, and `ssd` are TensorFlow-native adaptations. Before publication, compare their settings and outputs against the official repositories and state any implementation differences.
- Membership-inference values are empirical audit results, not formal privacy guarantees.

## RAM-safe execution notes

- Use **Runtime → Run all** after a clean Colab runtime is connected.
- Keep `FORCE_RERUN = False` to resume completed methods.
- Do not convert complete CIFAR arrays to `float32`; conversion occurs per mini-batch.
- The publication configuration uses three seeds and all enabled methods.
- If Colab disconnects, reconnect, run from the top, and completed model checkpoints will be reused.
